## Create Dataset Splits

In [ ]:
import time 

In [ ]:
print(f'START: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())}')
time.sleep(2)
print(f'END: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())}')


In [ ]:
import pandas as pd

In [ ]:
base_dataset = '../../datasets/core/sanitized-mbpp.json'
sampled_dataset = '../../datasets/core/sanitized-mbpp-sample-100.json'

In [ ]:
# read jsonl files
df = pd.read_json(base_dataset)
df.head()

In [ ]:
# create a reproducable sample of 100 rows
SEED_VALUE = 1204

df_sample = df.sample(n=100, random_state=SEED_VALUE)
df_sample.to_json(sampled_dataset, orient='records', lines=True)   

In [ ]:
sample_df = pd.read_json(sampled_dataset, lines=True)

In [ ]:
sample_df.columns

In [ ]:
lines = []
for _, row in sample_df.iterrows():
    lines.append(len(row['code']))
    print(len(row['code']))
    print("\n\n")

print(f"Max length: {max(lines)}")
print(f"Min length: {min(lines)}")
print(f"Avg length: {sum(lines)/len(lines)}")

In [ ]:
def load_model(model_id=default_model):
    
    model_id = model_id
    # model_id = "Qwen/Qwen2.5-Coder-14B-Instruct"
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",    # will spread across your 4 GPUs
        torch_dtype="auto"
    )

    return model, tokenizer

def genearate_by_qwen(model, tokenizer, messages, eos_ids=default_eos_ids):

    response = None
    # eos_ids = [151645,151644,151645,151646,151647,151648,151649,151650,151651,151652,151653,151654,151655,151656]
    eos_ids = eos_ids
    
    messages = messages
    
    inputs = tokenizer.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
    ).to(model.device)
    
    outputs = model.generate(
        **inputs, 
        eos_token_id=eos_ids,  
        pad_token_id=tokenizer.eos_token_id,   
        max_new_tokens=4096,
    )
    
    text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

    code_snippet = re.search(r'```python(.*?)```', text, re.DOTALL)

    try:
        
        if code_snippet:
            response = code_snippet.group(1).strip()
        elif 'json' in text.lower():
            json_snippet = re.search(r'```json(.*?)```', text, re.DOTALL)
            if json_snippet:
                response = json_snippet.group(1).strip()
        else:
            response = text.strip()

    except:
        print("ERROR HERE///")

    # print(f"Generated Response ----: \n {response}")
    return response

In [ ]:
def execute(record, model_type="qwen"):
    prompt_text = record["prompt"]
    test_cases = "\n".join(record["test_list"]) 

    template = PROMPTS["initials"]
    full_prompt = template.format(prompt_text=prompt_text, test_cases=test_cases)

    # Handle different model types
    if model_type.lower() in ["mistral", "mixtral"]:
        # For Mistral models: combine system prompt with user prompt
        combined_prompt = f"{SYSTEM_PROMPT_BASELINE}\n\n{full_prompt}"
        messages = [
            {
                "role": "user", 
                "content": combined_prompt
            },
        ]
    else:
        # For other models (Qwen, etc.): use separate system role
        messages = [
            {
                "role": "system", 
                "content": SYSTEM_PROMPT_BASELINE
            },
            {
                "role": "user", 
                "content": full_prompt
            },
        ]

    # Non-streaming call using the new Client API
    resp = genearate_by_qwen(model=model, tokenizer=tokenizer, messages=messages)

    return resp

In [ ]:
# Example usage for different models:

# For Mistral models
# resp = execute(record, model_type="mistral")

# For Qwen or other models (default)
# resp = execute(record, model_type="qwen")
# resp = execute(record)  # defaults to qwen behavior

## Calculate Confusion Matrix

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../../results/raw/exp_1_during_gen_v2_100.csv')
df.head()

In [ ]:
# print("# A code Wrongly identified as WM (FP): ", df['original_is_watermarked'].sum())
# print("# WM code Correctly identified as WM (TP): ", df['generated_is_watermarked'].sum())

# print("# A code Correctly identified as A (FN):", df.shape[0] - df['original_is_watermarked'].sum())
# print("# WM code Wrongly identified as A (TN): ", df.shape[0] - df['generated_is_watermarked'].sum())

# # print("# of code: ", df.shape[0])

In [ ]:
tp = df['generated_is_watermarked'].sum()   # WM correctly flagged
fn = df.shape[0] - tp                       # WM missed
fp = df['original_is_watermarked'].sum()    # A falsely flagged
tn = df.shape[0] - fp                       # A correctly ignored


In [ ]:
print(f"Confusion Matrix:")
print(f"       P-W   P-N")
print(f"A-W  {tp:4d} {fn:4d}")
print(f"A-N  {fp:4d} {tn:4d}")

In [ ]:
# Build ground truth labels
y_true = [0]*len(df) + [1]*len(df)

# Build binary predictions
y_pred = df['original_is_watermarked'].tolist() + df['generated_is_watermarked'].tolist()

# Build continuous scores
y_score = df['original_z_score'].tolist() + df['generated_z_score'].tolist()

print("TRUE: ", len(y_true))

In [ ]:
def metrics_from_confusion(tp: int, tn: int, fp: int, fn: int):
    """
    Returns point metrics from a single-threshold confusion matrix.
    NOTE: AUROC cannot be derived from a single confusion matrix.
    """
    import math

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else float('nan')

    precision = tp / (tp + fp) if (tp + fp) else float('nan')     # PPV
    recall    = tp / (tp + fn) if (tp + fn) else float('nan')     # TPR / sensitivity
    specificity = tn / (tn + fp) if (tn + fp) else float('nan')   # TNR

    if precision is None or recall is None or math.isnan(precision) or math.isnan(recall) or (precision + recall) == 0:
        f1 = float('nan')
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "auroc": None,                  # not computable from a single CM
        "roc_points": None              # not computable from a single CM
    }


metrics_from_confusion(tp, tn, fp, fn)

In [ ]:
# Build ground truth labels
y_true = [0]*len(df) + [1]*len(df)

# Build binary predictions
y_pred = df['original_is_watermarked'].tolist() + df['generated_is_watermarked'].tolist()

# Build continuous scores (if available)
y_score = df['original_z_score'].tolist() + df['generated_z_score'].tolist()

print("GROUND TRUTH LABELS (y_true): ", sum((y_true)))
print("PREDICTED LABELS (y_pred): ", sum(y_pred))


In [ ]:
def evaluate_watermark_detector(
    df,
    col_orig_pred="original_is_watermarked",
    col_gen_pred="generated_is_watermarked",
    col_orig_score="original_z_score",
    col_gen_score="generated_z_score",
    plot=False,
):
    """
    Build y_true / y_pred / y_score from your wide-format df and compute:
    TP, TN, FP, FN, accuracy, precision, recall, F1, specificity, AUROC, ROC points.

    Ground truth convention:
      - 0 = original (non-watermarked)
      - 1 = generated (watermarked)
    """
    import math
    import numpy as np

    n = len(df)
    if n == 0:
        raise ValueError("Empty dataframe.")

    # Ground truth labels: first originals (0), then generated (1)
    y_true = np.array([0]*n + [1]*n, dtype=int)

    # Binary predictions (your detector’s decisions at Z_THRESHOLD)
    y_pred = np.array(
        df[col_orig_pred].tolist() + df[col_gen_pred].tolist(),
        dtype=int
    )

    # Continuous scores (z-scores) for ROC/AUROC
    if col_orig_score in df.columns and col_gen_score in df.columns:
        y_score = np.array(
            df[col_orig_score].tolist() + df[col_gen_score].tolist(),
            dtype=float
        )
    else:
        y_score = None  # ROC/AUROC unavailable without scores

    # ---- Confusion matrix (using ground truth) ----
    # Positives = generated rows (second half)
    # Negatives = original rows (first half)
    tp = int((y_true == 1).astype(int)[(y_pred == 1)].sum())
    fn = int((y_true == 1).astype(int)[(y_pred == 0)].sum())
    fp = int((y_true == 0).astype(int)[(y_pred == 1)].sum())
    tn = int((y_true == 0).astype(int)[(y_pred == 0)].sum())

    total = tp + tn + fp + fn

    def safe_div(a, b):
        return a / b if b else float("nan")

    accuracy    = safe_div(tp + tn, total)
    precision   = safe_div(tp, tp + fp)           # PPV
    recall      = safe_div(tp, tp + fn)           # TPR / sensitivity
    specificity = safe_div(tn, tn + fp)           # TNR
    f1 = (2 * precision * recall / (precision + recall)
          if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0)
          else float("nan"))

    # ---- ROC + AUROC from scores (if provided) ----
    roc = {"fpr": None, "tpr": None, "thresholds": None, "auroc": None}

    if y_score is not None:
        # Need both classes present
        pos_mask = (y_true == 1)
        neg_mask = (y_true == 0)
        n_pos = int(pos_mask.sum())
        n_neg = int(neg_mask.sum())

        if n_pos > 0 and n_neg > 0:
            # Sort by score desc
            order = np.argsort(-y_score, kind="mergesort")
            y_true_sorted = y_true[order]
            y_score_sorted = y_score[order]

            # Distinct thresholds
            distinct = np.where(np.diff(y_score_sorted))[0]
            thresholds = np.array([1.96])

            # Cumulative counts
            tps = np.cumsum((y_true_sorted == 1).astype(int))
            fps = np.cumsum((y_true_sorted == 0).astype(int))

            tpr = np.r_[0.0, tps[distinct] / n_pos, 1.0]
            fpr = np.r_[0.0, fps[distinct] / n_neg, 1.0]

            auroc = float(np.trapz(y=tpr, x=fpr))

            roc = {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist(),
                "thresholds": thresholds.tolist(),
                "auroc": auroc,
            }

            print("ROC: ", roc['thresholds'])

            if plot:
                import matplotlib.pyplot as plt
                plt.figure()
                plt.plot(fpr, tpr, label=f"ROC (AUROC={auroc:.3f})")
                plt.plot([0,1], [0,1], linestyle="--", label="Chance")
                plt.xlabel("False Positive Rate")
                plt.ylabel("True Positive Rate")
                plt.title("ROC Curve")
                plt.legend()
                plt.show()

    return {
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc": roc,
        "y_true": y_true,     # returned for convenience if you want to reuse
        "y_pred": y_pred,
        "y_score": y_score,
    }

x = evaluate_watermark_detector(df, plot=True)

## Process CodeSearchNet & HumanEval - Python Dataset

In [ ]:
# !pip install autonotebook

In [ ]:
from datasets import load_dataset
import pandas as pd
from tqdm.notebook import tqdm

ds = load_dataset("Nan-Do/code-search-net-python")

# Sample 1000 examples from the train split (assuming 'train' is the split)
sampled_ds = ds['train'].shuffle(seed=42).select(range(1000))

# Convert to pandas DataFrame for easy JSONL export
df = pd.DataFrame(sampled_ds)

# Save as JSONL

In [ ]:
df.to_json('../../datasets/core/code-search-net-python-sample-1k.jsonl', orient='records', lines=True)


In [ ]:
df.rename(columns={"summary": "prompt"}, inplace=True)

In [ ]:
for i in range(10):
    print(f"[{i+1}]: {df.iloc[i]['code']}")

In [ ]:
from datasets import load_dataset
import pandas as pd

from datasets import load_dataset
# Languages: "python", "js", "java", "go", "cpp", "rust"
ds = load_dataset("bigcode/humanevalpack", "python")["test"]

In [ ]:
ds.column_names

In [ ]:
for i in range(10):
    print(f"[{i+1}]: ----\n{ds[i]['docstring']}")

In [ ]:
df = pd.DataFrame(sampled_ds)
df.to_json('../../datasets/core/humanevalplus-python-164.json', orient='records', lines=True)

## Handle Parquate Dataset

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet('../../datasets/test-00000-of-00001-5c45fa6e45572491.parquet')
df.shape

In [ ]:
df.columns

In [ ]:
for idx, row in df.iterrows():
    print(f"[{idx+1}]: {row['fields']}\n\n")
    if idx == 10:
        break

In [ ]:
df.to_json('../../datasets/core/classeval-100.json', orient='records', lines=True)

In [ ]:
# import json
# import pandas as pd

# # Load the JSONL file
# data = []
# with open('../../datasets//hmcorp.jsonl', 'r') as f:
#     for line in f:
#         js = json.loads(line.strip())
#         data.append(js)

# # Transform to new format
# transformed = []
# for item in data:
#     idx = item['index']
#     if item['label'] == 0:
#         human_code = item['code']
#         ai_code = item['contrast']
#     else:
#         human_code = item['contrast']
#         ai_code = item['code']
#     transformed.append({'id': idx, 'human_code': human_code, 'ai_code': ai_code})

# # Create DataFrame and save to CSV
# df = pd.DataFrame(transformed)
# df.to_json('../../datasets/core/redefined_hmcorp.jsonl', lines=True, orient="records")
# print("Dataset redefined and saved to 'redefined_hmcorp.jsonl'")

## Evaluate Generated Code Quality

In [ ]:
import pandas as pd
sample_df = pd.read_json('../../datasets/core/sanitized-mbpp-sample-100.json', lines=True)
sample_df.shape

In [ ]:
import tokenize
import io
import matplotlib.pyplot as plt

# Assuming sample_df is available from previous cells
# Compute character lengths and token lengths
char_lengths = []
token_lengths = []

for _, row in sample_df.iterrows():
    code = row['code']
    char_lengths.append(len(code))
    # Manual tokenization using Python's tokenize module
    try:
        tokens = list(tokenize.generate_tokens(io.StringIO(code).readline))
        token_lengths.append(len(tokens))
    except tokenize.TokenError:
        # Handle tokenization errors by falling back to character count or skipping
        token_lengths.append(len(code.split()))  # Simple fallback: word count

# Add to dataframe for easy access
sample_df['char_length'] = char_lengths
sample_df['token_length'] = token_lengths

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Character length histogram
axes[0].hist(char_lengths, bins=20, alpha=0.7, color='blue')
axes[0].set_title('Distribution of Code Character Lengths')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

# Token length histogram
axes[1].hist(token_lengths, bins=20, alpha=0.7, color='green')
axes[1].set_title('Distribution of Code Token Lengths')
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Character Length - Max: {max(char_lengths)}, Min: {min(char_lengths)}, Avg: {sum(char_lengths)/len(char_lengths):.2f}")
print(f"Token Length - Max: {max(token_lengths)}, Min: {min(token_lengths)}, Avg: {sum(token_lengths)/len(token_lengths):.2f}")

In [ ]:
import os 

char_lengths = []
token_lengths = []

dir = '../../output/qwen14_exp1-1_during_gen_v4_100_mbpp'

for item in os.listdir(dir):

    # read file content
    with open(os.path.join(dir, item), 'r') as file:
        content = file.read()
        char_lengths.append(len(content))
        
         # Manual tokenization using Python's tokenize module
        try:
            tokens = list(tokenize.generate_tokens(io.StringIO(content).readline))
            token_lengths.append(len(tokens))
        except tokenize.TokenError:
            # Handle tokenization errors by falling back to character count or skipping
            token_lengths.append(len(content.split()))  # Simple fallback: word count


# Plot histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Character length histogram
axes[0].hist(char_lengths, bins=20, alpha=0.7, color='blue')
axes[0].set_title('Distribution of Code Character Lengths')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

# Token length histogram
axes[1].hist(token_lengths, bins=20, alpha=0.7, color='green')
axes[1].set_title('Distribution of Code Token Lengths')
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Character Length - Max: {max(char_lengths)}, Min: {min(char_lengths)}, Avg: {sum(char_lengths)/len(char_lengths):.2f}")
print(f"Token Length - Max: {max(token_lengths)}, Min: {min(token_lengths)}, Avg: {sum(token_lengths)/len(token_lengths):.2f}")


In [ ]:
import pandas as pd

df = pd.read_json('../../datasets/core/humanevalplus-python-164.json', lines=True)
df.columns

In [ ]:
import pprint

pprint.pprint(df.iloc[0]['original_string'])

## Fix Detection Method

In [ ]:
from scipy.stats import norm

# make thresholds consistent
# P_THRESHOLD = P_THRESHOLD  # set once
Z_THRESHOLD = 2.120
P_THRESHOLD = norm.sf(Z_THRESHOLD)  # ensure consistency (one-sided)

print("Z_THRESHOLD: ", Z_THRESHOLD)
print("P_THRESHOLD: ", P_THRESHOLD)

### Save dataset code

In [ ]:
import pandas as pd
# df = pd.read_json('../../datasets/core/sanitized-mbpp-sample-100.json', lines=True)
df = pd.read_parquet('../../datasets/core/human_eval_164.parquet')
df.columns

In [ ]:
# write all the code snippets to individual files
import textwrap
import os
output_dir = '../../datasets/core/humaneval-164-codes'
os.makedirs(output_dir, exist_ok=True)

for idx, row in df.iterrows():
    code_snippet = row['canonical_solution']
    tid = row['task_id'].split('/')[-1]
    file_path = os.path.join(output_dir, f'{tid}.py')
    with open(file_path, 'w') as f:
        f.write(textwrap.dedent(code_snippet))

### Testing dataset

In [ ]:
import pandas as pd
import os 

DATASET_PATH = f"../../datasets/core/sanitized-mbpp-sample-100.json"
df = pd.read_json(DATASET_PATH, lines=True)
df.shape

In [ ]:
ids = df['task_id'].tolist()
print("Task IDs: ", ids)

In [ ]:
files = []
folder_path = '../../output/gemini_exp3-1_during_gen_v2_100_mbpp'
files = os.listdir(folder_path)
generated = [int(f.split('.')[0]) for f in files if f.endswith('.py')]
print("Generated files: ", generated)

In [ ]:
actual_id = set(ids)
done_id = set(generated)
missing_id = actual_id - done_id
print("Missing IDs: ", sorted(missing_id))
print("Number of missing IDs: ", len(missing_id))

In [ ]:
df.shape

In [ ]:
df = df[df['task_id'].isin(missing_id)]

In [ ]:
df.shape

## Calculate Code Metrics

In [ ]:
# !pip install radon -q

In [ ]:
import pandas as pd 

sample_df = pd.read_parquet('/home/fahad/Documents/PROJECTS/promptmark/datasets/human_eval_164.parquet')
sample_df = sample_df[10:15]
sample_df['code'] = sample_df['prompt'] + '\n' + sample_df['canonical_solution']
sample_df

In [ ]:
print(sample_df.iloc[3]['code'])

In [ ]:
from radon.raw import analyze as raw_analyze
from radon.complexity import cc_visit
from radon.metrics import mi_visit
import textwrap

def extract_radon_metrics(code):
    normalized_code = textwrap.dedent(code).strip()
    raw = raw_analyze(normalized_code)
    blocks = cc_visit(normalized_code)
    max_cc = max((block.complexity for block in blocks), default=0)
    avg_cc = sum(block.complexity for block in blocks) / len(blocks) if blocks else 0.0
    return {
        "loc": raw.loc, # lines of code
        "lloc": raw.lloc, # logical lines of code
        "sloc": raw.sloc, # source lines of code
        "comments": raw.comments, # number of comment lines
        "blank": raw.blank, # number of blank lines
        "max_cc": max_cc, # maximum cyclomatic complexity
        "avg_cc": avg_cc, # average cyclomatic complexity
        "mi": mi_visit(normalized_code, True), # maintainability index
        "cc_blocks": len(blocks), # number of code blocks analyzed
    }

metrics = []
for _, row in sample_df.iterrows():
    metrics.append({"task_id": row["task_id"], **extract_radon_metrics(row["code"])})
    
metrics_df = pd.DataFrame(metrics)
metrics_df = metrics_df.sort_values("avg_cc", ascending=False).head()

# print("Summary:")
# print(metrics_df[["loc", "lloc", "sloc", "avg_cc", "mi"]].describe().loc[["mean", "min", "max"]])

In [ ]:
gen_df = pd.read_csv('generated_code_human_eval_1.csv')
# gen_df['task_id'] = gen_df[]
# gen_df

In [ ]:
metrics = []
for _, row in gen_df.iterrows():

    try:
        metrics.append({
            "task_id": row["task_id"], 
            "model": row["model"], 
            "version": row['version'],
            **extract_radon_metrics(row["code"])
        })
    except: 
        pass 

print(len(metrics))
    
metrics_df2 = pd.DataFrame(metrics)
# print("MX SHAPE: ", metrics_df2.shape)
metrics_df2 = metrics_df2.sort_values("avg_cc", ascending=False)

# print("Summary: ", metrics_df2.shape)
# print(metrics_df2[["loc", "lloc", "sloc", "avg_cc", "mi"]].describe().loc[["mean", "min", "max"]])

In [ ]:
metrics_df.iloc[4]

In [ ]:
metrics_df2.sort_values('model')